# Regresión Lineal y Bootstrap

|                |   |
:----------------|---|
| **Nombre**     David Alejandro Rangel Rodríguez|   |
| **Fecha**     27 de abril del 2026 |   |
| **Expediente*756203* |   |

In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

df = pd.read_excel("Motor Trend Car Road Tests.xlsx")
y = df["mpg"].values
X_raw = df[["hp", "qsec"]].values
X = sm.add_constant(X_raw)          
n, k = X.shape  

In [4]:
# REGRESIÓN OLS CLÁSICA

modelo = sm.OLS(y, X).fit()
print(modelo.summary())
 
beta_ols  = modelo.params          # [β0, β1, β2]
se_ols    = modelo.bse             # errores estándar
t_crit    = stats.t.ppf(0.975, df=n - k)   # t* con α=0.05, dos colas
 
# IC 95 % con t-distribution
ic_ols_lower = beta_ols - t_crit * se_ols
ic_ols_upper = beta_ols + t_crit * se_ols
 
nombres = ["β0 (Intercept)", "β1 (hp)", "β2 (qsec)"]
print("\n── Intervalos de Confianza 95% (OLS) ──")
print(f"{'Parámetro':<20} {'Estimado':>10} {'IC Inf':>10} {'IC Sup':>10}")
print("-" * 52)
for i, nom in enumerate(nombres):
    print(f"{nom:<20} {beta_ols[i]:>10.4f} {ic_ols_lower[i]:>10.4f} {ic_ols_upper[i]:>10.4f}")
 

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:                Thu, 30 Apr 2026   Prob (F-statistic):           4.18e-07
Time:                        16:42:47   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.0

In [5]:
# BOOTSTRAP (B=1000 muestras con reemplazo, m=n)
B      = 1000
m      = n                         # tamaño de muestra = tamaño original
rng    = np.random.default_rng(42)
betas_boot = np.zeros((B, k))      # guarda β0, β1, β2 de cada muestra
 
for b in range(B):
    idx          = rng.choice(n, size=m, replace=True)
    X_b, y_b    = X[idx], y[idx]
    res_b        = sm.OLS(y_b, X_b).fit()
    betas_boot[b] = res_b.params
 
# 3a. Media y desviación estándar de los betas bootstrap
beta_boot_mean = betas_boot.mean(axis=0)
beta_boot_std  = betas_boot.std(axis=0, ddof=1)
 
# 3b. IC bootstrap – método percentil 2.5 / 97.5
ic_boot_lower = np.percentile(betas_boot, 2.5, axis=0)
ic_boot_upper = np.percentile(betas_boot, 97.5, axis=0)
 
print(f"\n{'Parámetro':<20} {'Media Boot':>12} {'Std Boot':>12} {'IC Inf (2.5%)':>14} {'IC Sup (97.5%)':>15}")
print("-" * 75)
for i, nom in enumerate(nombres):
    print(f"{nom:<20} {beta_boot_mean[i]:>12.4f} {beta_boot_std[i]:>12.4f} "
          f"{ic_boot_lower[i]:>14.4f} {ic_boot_upper[i]:>15.4f}")


Parámetro              Media Boot     Std Boot  IC Inf (2.5%)  IC Sup (97.5%)
---------------------------------------------------------------------------
β0 (Intercept)            49.7478      10.9492        30.6417         71.8820
β1 (hp)                   -0.0877       0.0168        -0.1215         -0.0580
β2 (qsec)                 -0.9477       0.5301        -2.0846         -0.0686


In [6]:
# COMPARACIÓN OLS vs BOOTSTRAP
print(f"\n{'Parámetro':<20} {'OLS β':>8} {'Boot μ':>8} {'OLS SE':>8} {'Boot SD':>8} "
      f"{'OLS IC inf':>11} {'OLS IC sup':>11} {'Boot IC inf':>12} {'Boot IC sup':>12}")
print("-" * 100)
for i, nom in enumerate(nombres):
    print(f"{nom:<20} {beta_ols[i]:>8.4f} {beta_boot_mean[i]:>8.4f} "
          f"{se_ols[i]:>8.4f} {beta_boot_std[i]:>8.4f} "
          f"{ic_ols_lower[i]:>11.4f} {ic_ols_upper[i]:>11.4f} "
          f"{ic_boot_lower[i]:>12.4f} {ic_boot_upper[i]:>12.4f}")
 


Parámetro               OLS β   Boot μ   OLS SE  Boot SD  OLS IC inf  OLS IC sup  Boot IC inf  Boot IC sup
----------------------------------------------------------------------------------------------------
β0 (Intercept)        48.3237  49.7478  11.1033  10.9492     25.6149     71.0325      30.6417      71.8820
β1 (hp)               -0.0846  -0.0877   0.0139   0.0168     -0.1131     -0.0561      -0.1215      -0.0580
β2 (qsec)             -0.8866  -0.9477   0.5346   0.5301     -1.9799      0.2068      -2.0846      -0.0686


In [8]:
# FIGURA: distribución bootstrap + IC
fig = plt.figure(figsize=(15, 11))
fig.suptitle("Regresión OLS vs Bootstrap — mpg ~ β0 + β1·hp + β2·qsec",
             fontsize=14, fontweight="bold", y=1.01)
 
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
 
colores = ["#2196F3", "#4CAF50", "#FF5722"]
etiquetas_cortas = ["β0", "β1 (hp)", "β2 (qsec)"]
 
# Fila 0: histogramas bootstrap con IC
for i in range(k):
    ax = fig.add_subplot(gs[0, i])
    ax.hist(betas_boot[:, i], bins=35, color=colores[i], alpha=0.75, edgecolor="white")
    ax.axvline(beta_ols[i],       color="black",  lw=2,   ls="-",  label="OLS β")
    ax.axvline(ic_ols_lower[i],   color="black",  lw=1.5, ls="--", label="IC OLS 95%")
    ax.axvline(ic_ols_upper[i],   color="black",  lw=1.5, ls="--")
    ax.axvline(beta_boot_mean[i], color="orange", lw=2,   ls="-",  label="Boot media")
    ax.axvline(ic_boot_lower[i],  color="red",    lw=1.5, ls=":",  label="IC Boot 95%")
    ax.axvline(ic_boot_upper[i],  color="red",    lw=1.5, ls=":")
    ax.set_title(f"Bootstrap {etiquetas_cortas[i]}", fontsize=11)
    ax.set_xlabel("Valor del parámetro")
    ax.set_ylabel("Frecuencia")
    if i == 0:
        ax.legend(fontsize=7, loc="upper right")
 
# Fila 1: comparación IC lado a lado
ax_ic = fig.add_subplot(gs[1, :])
y_pos   = np.arange(k)
ancho   = 0.35
 
# barras de longitud del IC
long_ols  = ic_ols_upper  - ic_ols_lower
long_boot = ic_boot_upper - ic_boot_lower
 
bars1 = ax_ic.barh(y_pos - ancho/2, long_ols,  ancho, color="#2196F3", alpha=0.8, label="Amplitud IC OLS")
bars2 = ax_ic.barh(y_pos + ancho/2, long_boot, ancho, color="#FF5722", alpha=0.8, label="Amplitud IC Bootstrap")
 
ax_ic.set_yticks(y_pos)
ax_ic.set_yticklabels(etiquetas_cortas, fontsize=11)
ax_ic.set_xlabel("Amplitud del intervalo de confianza 95%")
ax_ic.set_title("Comparación de amplitudes de IC — OLS vs Bootstrap")
ax_ic.legend()
 
# anotar valores
for bar in bars1:
    ax_ic.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
               f"{bar.get_width():.4f}", va="center", fontsize=9)
for bar in bars2:
    ax_ic.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
               f"{bar.get_width():.4f}", va="center", fontsize=9)
 
plt.savefig("regresion_bootstrap.png", dpi=150, bbox_inches="tight")
plt.close()

## Aggregating

|                |   |
:----------------|---|
| **Nombre**     David Alejandro Rangel Rodríguez|   |
| **Fecha**     30 de abril del 2026 |   |
| **Expediente** 756203* |   |

In [11]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X_full = df.drop(columns=["model", "mpg"])
y = df["mpg"]

features = X_full.columns.tolist()
n = len(y)

# Listas principales
resultados = []
vars_ = []   # almacena las columnas de cada iteración
models = []  # almacena cada modelo entrenado

for i in range(1000):

    # 1) Seleccionar 3 columnas al azar (sin reemplazo)
    var_ = np.random.choice(features, size=3, replace=False).tolist()
    vars_.append(var_)

    X = X_full[var_]

    # 2) Split 50% train / 50% test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.50, random_state=i
    )

    # 3) Ajustar modelo de regresión lineal
    model = LinearRegression()
    model.fit(X_train, y_train)
    models.append(model)

    # 4) Predicción sobre el conjunto test → ŷ_test
    y_hat_test = model.predict(X_test)

    # 5) R² sobre test
    r2_test = r2_score(y_test, y_hat_test)

    resultados.append({
        "modelo_id" : i + 1,
        "columnas"  : var_,
        "R2_test"   : round(r2_test, 6),
    })

# Resultados
resultados_df = pd.DataFrame(resultados)

print("=" * 55)
print(f"  Total de modelos entrenados : {len(resultados_df)}")
print(f"  R² test  —  media  : {resultados_df['R2_test'].mean():.4f}")
print(f"  R² test  —  std    : {resultados_df['R2_test'].std():.4f}")
print(f"  R² test  —  min    : {resultados_df['R2_test'].min():.4f}")
print(f"  R² test  —  max    : {resultados_df['R2_test'].max():.4f}")
print("=" * 55)

# Top 5 mejores modelos
top5 = resultados_df.nlargest(5, "R2_test")
print("\nTop 5 modelos por R² test:")
print(top5[["modelo_id", "columnas", "R2_test"]].to_string(index=False))

# Ejemplo detallado: mejor modelo 
mejor     = top5.iloc[0]
mejor_idx = int(mejor["modelo_id"]) - 1   # índice 0-based en las listas


cols_mejor  = vars_[mejor_idx]
model_final = models[mejor_idx]

X_m = X_full[cols_mejor]

X_train, X_test, y_train, y_test = train_test_split(
    X_m, y, test_size=0.50, random_state=mejor_idx
)

y_hat_test = model_final.predict(X_test)

print(f"\n--- Detalle del mejor modelo (ID {int(mejor['modelo_id'])}) ---")
print(f"Columnas usadas : {cols_mejor}")
print(f"R² test         : {mejor['R2_test']:.4f}")
print(f"\n{'Obs':>4}  {'y_test':>8}  {'y_hat_test':>10}")
print("-" * 28)
for obs, (yt, yh) in enumerate(zip(y_test, y_hat_test), 1):
    print(f"{obs:>4}  {yt:>8.2f}  {yh:>10.4f}")

  Total de modelos entrenados : 1000
  R² test  —  media  : 0.5930
  R² test  —  std    : 0.2358
  R² test  —  min    : -1.7681
  R² test  —  max    : 0.8779

Top 5 modelos por R² test:
 modelo_id        columnas  R2_test
       149  [hp, qsec, wt] 0.877890
       485 [wt, gear, cyl] 0.875119
       268    [vs, wt, am] 0.862282
       663    [hp, wt, vs] 0.862183
       210  [disp, hp, wt] 0.859441

--- Detalle del mejor modelo (ID 149) ---
Columnas usadas : ['hp', 'qsec', 'wt']
R² test         : 0.8779

 Obs    y_test  y_hat_test
----------------------------
   1     30.40     28.2516
   2     17.80     20.3141
   3     15.20     17.6407
   4     22.80     25.9128
   5     30.40     26.7560
   6     26.00     23.8651
   7     19.20     16.3743
   8     16.40     15.7295
   9     17.30     17.4407
  10     15.20     18.5461
  11     13.30     14.3938
  12     24.40     22.8165
  13     27.30     27.1833
  14     15.80     16.3188
  15     21.40     21.9269
  16     10.40     10.0558
